<a href="https://colab.research.google.com/github/gonmpa/Laboratorio_2.12_Multi_Layer_Perceptron/blob/main/MLP_2_12_Gonzalo_Pino.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 2.12 - Laboratorio: Implementación de Redes Neuronales (MLP + MLflow)
**Estudiante:** Gonzalo Pino
**Número de lista:** 33
**Fecha:** 10 de Noviembre de 2026

In [1]:
!pip install -q mlflow pyngrok

In [2]:
import pandas as pd
import numpy as np
import random
import kagglehub
import sklearn
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# FIJA TU SEMILLA AQUÍ (Reemplaza el 10 por tu número de lista)
SEED = 33
random.seed(SEED)
np.random.seed(SEED)

print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"MLflow version: {mlflow.__version__}")

Pandas version: 2.2.3
Scikit-learn version: 1.6.1
MLflow version: 3.18.0


In [3]:
# Descargar el dataset
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
print("Ruta del dataset:", path)

# Cargar el CSV (ajustar el nombre del archivo si es necesario según lo que descargue)
df = pd.read_csv(f"{path}/data.csv")

print("Shape:", df.shape)
df.info()

# Verificar nulos y eliminar columnas innecesarias
print("\nValores nulos por columna:\n", df.isna().sum()[df.isna().sum() > 0])
df = df.drop(columns=['Unnamed: 32', 'id'])

# Codificar la etiqueta diagnosis: M -> 1, B -> 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Balance de clases
balance = df['diagnosis'].value_counts()
print("\nBalance de clases:\n", balance)

# Evidencia de escala
min_radius = df['radius_mean'].min()
max_radius = df['radius_mean'].max()
min_area = df['area_mean'].min()
max_area = df['area_mean'].max()
min_smooth = df['smoothness_mean'].min()
max_smooth = df['smoothness_mean'].max()

print(f"\nRango radius_mean: {min_radius} a {max_radius}")
print(f"Rango area_mean: {min_area} a {max_area}")
print(f"Rango smoothness_mean: {min_smooth} a {max_smooth}")
print(f"El valor máximo de area_mean es {max_area / max_smooth:.2f} veces mayor que el máximo de smoothness_mean.")

100%|██████████| 48.6k/48.6k [00:00<00:00, 26.5MB/s]

Extracting files...
Ruta del dataset: /root/.cache/kagglehub/datasets/uciml/breast-cancer-wisconsin-data/versions/2
Shape: (569, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  r

**Balance de clases registrado:**
* Benignos (0): 357
* Malignos (1): 212

In [4]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# Partición estratificada (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

# Comprobación de la estratificación
print("Proporción en Train:\n", y_train.value_counts(normalize=True))
print("\nProporción en Test:\n", y_test.value_counts(normalize=True))

Proporción en Train:
 diagnosis
0    0.626374
1    0.373626
Name: proportion, dtype: float64

Proporción en Test:
 diagnosis
0    0.631579
1    0.368421
Name: proportion, dtype: float64


In [9]:
import os, subprocess, time, requests
from google.colab import userdata
from pyngrok import ngrok

# 1) Servidor de seguimiento con SQLite como backend store
env = dict(os.environ,
           MLFLOW_SERVER_ALLOWED_HOSTS="*",
           MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*")

subprocess.Popen(["mlflow", "server",
                  "--backend-store-uri", "sqlite:///mlflow.db",
                  "--host", "127.0.0.1", "--port", "5000"], env=env)

# 2) Esperar a que el servidor responda
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=1).ok:
            break
    except (requests.ConnectionError, requests.Timeout):  # <-- CORRECCIÓN AQUÍ
        pass
    time.sleep(1)

# 3) Túnel público con ngrok
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill() # cierra túneles anteriores
tunel = ngrok.connect(5000)
print("MLflow UI:", tunel.public_url)

MLflow UI: https://rely-parsley-eradicate.ngrok-free.dev


In [10]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("IEI097-L2.12-Pino")

2026/11/11 01:48:03 INFO mlflow.tracking.fluent: Experiment with name 'IEI097-L2.12-Pino' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1794361683148, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794361683148, lifecycle_stage='active', name='IEI097-L2.12-Pino', tags={}, trace_location=None, workspace='default'>

### Hipótesis para Runs A y B
**¿El escalado mejorará, empeorará o no cambiará el resultado?**
El escalado mejorará sustancialmente el rendimiento del modelo. Como evidenciamos en el paso 2, la variable `area_mean` tiene un valor máximo que es 15,306 veces mayor que `smoothness_mean`. Dado que el perceptrón multicapa ajusta sus pesos mediante descenso del gradiente, los atributos de gran magnitud dominan la suma ponderada y pueden saturar las funciones de activación, haciendo que la convergencia sea inestable o requiera más iteraciones de las permitidas (`max_iter=200`). Al estandarizar las variables (`StandardScaler`), todas contribuyen equitativamente en la misma escala, permitiendo una optimización más rápida y precisa.

In [11]:
def correr_experimento(run_name, escalado, hidden_layers, activation, alpha=0.0001, max_iter=200):
    with mlflow.start_run(run_name=run_name):
        # 1. Configuración del Pipeline
        pasos = []
        if escalado:
            pasos.append(('scaler', StandardScaler()))

        mlp = MLPClassifier(hidden_layer_sizes=hidden_layers,
                            activation=activation,
                            alpha=alpha,
                            max_iter=max_iter,
                            random_state=SEED)
        pasos.append(('mlp', mlp))

        modelo = Pipeline(pasos)

        # 2. Entrenamiento
        modelo.fit(X_train, y_train)

        # 3. Predicciones
        y_pred_train = modelo.predict(X_train)
        y_pred_test = modelo.predict(X_test)

        # 4. Métricas
        acc_train = accuracy_score(y_train, y_pred_train)
        acc_test = accuracy_score(y_test, y_pred_test)
        recall_test = recall_score(y_test, y_pred_test)
        f1_test = f1_score(y_test, y_pred_test)

        # Extraer iteraciones y curvas del MLP interno
        mlp_step = modelo.named_steps['mlp']
        iteraciones = mlp_step.n_iter_

        # 5. Registro en MLflow
        mlflow.set_tag("fase", "experimentos")

        mlflow.log_params({
            "arquitectura": str(hidden_layers),
            "activacion": activation,
            "alpha": alpha,
            "max_iter": max_iter,
            "random_state": SEED,
            "escalado": escalado,
            "semilla_particion": SEED
        })

        mlflow.log_metrics({
            "acc_train": acc_train,
            "acc_test": acc_test,
            "recall_maligna": recall_test,
            "f1": f1_test,
            "iteraciones": iteraciones
        })

        # Registrar la pérdida por época (step)
        for epoch, loss in enumerate(mlp_step.loss_curve_):
            mlflow.log_metric("train_loss", loss, step=epoch)

        # 6. Artefacto: Matriz de Confusión
        fig, ax = plt.subplots()
        cm = confusion_matrix(y_test, y_pred_test)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benigno', 'Maligno'])
        disp.plot(ax=ax, cmap='Blues')
        plt.title(f"Matriz de Confusión - {run_name}")

        imagen_cm = f"cm_{run_name}.png"
        plt.savefig(imagen_cm)
        mlflow.log_artifact(imagen_cm)
        plt.close()

        print(f"[{run_name}] Completado | Iteraciones: {iteraciones} | Acc Test: {acc_test:.4f} | Recall: {recall_test:.4f}")

# Ejecución de los 4 experimentos obligatorios
correr_experimento(run_name="Run_A", escalado=False, hidden_layers=(30,), activation='relu')
correr_experimento(run_name="Run_B", escalado=True, hidden_layers=(30,), activation='relu')
correr_experimento(run_name="Run_C", escalado=True, hidden_layers=(64, 32), activation='relu')
correr_experimento(run_name="Run_D", escalado=True, hidden_layers=(64, 32), activation='tanh')

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[Run_A] Completado | Iteraciones: 200 | Acc Test: 0.9298 | Recall: 0.8095
🏃 View run Run_A at: http://127.0.0.1:5000/#/experiments/1/runs/c01466b5b8584d989f8bb3999304b71c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[Run_B] Completado | Iteraciones: 200 | Acc Test: 0.9386 | Recall: 0.9048
🏃 View run Run_B at: http://127.0.0.1:5000/#/experiments/1/runs/e2c449e5b8634216ae6bbd5c37604cb0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_C] Completado | Iteraciones: 156 | Acc Test: 0.9474 | Recall: 0.8810
🏃 View run Run_C at: http://127.0.0.1:5000/#/experiments/1/runs/8fca2e10781e41f59b3f00a261e230b1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[Run_D] Completado | Iteraciones: 200 | Acc Test: 0.9298 | Recall: 0.8810
🏃 View run Run_D at: http://127.0.0.1:5000/#/experiments/1/runs/aa3557b217af4fda8ca7a9c6a574e061
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Fase 4: Semilla y Réplica
En esta fase comprobaremos el principio de replicabilidad ejecutando el mismo experimento (Run_B) utilizando tres semillas distintas (`SEED`, `SEED + 1`, `SEED + 2`).

In [12]:
# Ejecución de réplicas variando únicamente la semilla del modelo
for i in range(3):
    semilla_actual = SEED + i
    nombre_run = f"Run_Replica_Seed_{semilla_actual}"

    with mlflow.start_run(run_name=nombre_run):
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('mlp', MLPClassifier(hidden_layer_sizes=(30,),
                                 activation='relu',
                                 alpha=0.0001,
                                 max_iter=200,
                                 random_state=semilla_actual))
        ])

        pipeline.fit(X_train, y_train)

        y_pred_train = pipeline.predict(X_train)
        y_pred_test = pipeline.predict(X_test)

        acc_test = accuracy_score(y_test, y_pred_test)
        recall_test = recall_score(y_test, y_pred_test)
        f1_test = f1_score(y_test, y_pred_test)

        mlflow.set_tag("fase", "replica_semilla")

        mlflow.log_params({
            "arquitectura": "(30,)",
            "activacion": "relu",
            "alpha": 0.0001,
            "max_iter": 200,
            "random_state": semilla_actual,
            "escalado": True,
            "semilla_particion": SEED
        })

        mlflow.log_metrics({
            "acc_test": acc_test,
            "recall_maligna": recall_test,
            "f1": f1_test,
            "iteraciones": pipeline.named_steps['mlp'].n_iter_
        })

        print(f"[{nombre_run}] Completado | Seed: {semilla_actual} | Acc Test: {acc_test:.4f} | Recall: {recall_test:.4f}")

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[Run_Replica_Seed_33] Completado | Seed: 33 | Acc Test: 0.9386 | Recall: 0.9048
🏃 View run Run_Replica_Seed_33 at: http://127.0.0.1:5000/#/experiments/1/runs/ec1f6edd51a74b9baa6323950b7f1259
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[Run_Replica_Seed_34] Completado | Seed: 34 | Acc Test: 0.9386 | Recall: 0.9048
🏃 View run Run_Replica_Seed_34 at: http://127.0.0.1:5000/#/experiments/1/runs/52f47a33fc994f07b0dfa1d493f07cea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Replica_Seed_35] Completado | Seed: 35 | Acc Test: 0.9386 | Recall: 0.9048
🏃 View run Run_Replica_Seed_35 at: http://127.0.0.1:5000/#/experiments/1/runs/61928336409b4c1596a0d9fecd8b720d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


### Interpretación de Resultados: Fase 4 - Replicabilidad y Variación de Semilla

#### 1. Análisis de Replicabilidad Exacta
* **Réplica idéntica:** La primera corrida (`Run_Replica_Seed_33`) obtuvo exactamente los mismos resultados que el **Run_B** de la Fase 3 (**Accuracy: 0.9386** y **Recall: 0.9048**). Esto evidencia que al fijar la semilla inicial (`SEED = 33`), el proceso de partición y la inicialización de pesos del MLP se reproducen de manera 100% determinista.

#### 2. Comportamiento frente a la Variación de Semilla
* **Estabilidad del modelo:** Al variar la semilla a `34` y `35`, las métricas de evaluación se mantuvieron idénticas (**Accuracy: 0.9386** y **Recall: 0.9048**).
* **Sensibilidad a la inicialización:** En esta arquitectura (`hidden_layer_sizes=(30,)` con variables estandarizadas), la optimización mediante gradiente descendente converge hacia la misma frontera de decisión efectiva a pesar de los pequeños cambios en la inicialización aleatoria de los pesos iniciales de la red.
* **Límite de iteraciones:** Todas las ejecuciones alcanzaron el máximo permitido de 200 iteraciones (`ConvergenceWarning`), lo que indica que, bajo la configuración por defecto de `max_iter=200`, el optimizador detiene la búsqueda en el mismo punto de parada.

In [13]:
# Fase 6: Variabilidad con 5 semillas distintas usando la configuración del Run_C
resultados_semillas = []

for i in range(5):
    semilla_actual = SEED + i
    nombre_run = f"Run_Semilla_{semilla_actual}"

    with mlflow.start_run(run_name=nombre_run):
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('mlp', MLPClassifier(hidden_layer_sizes=(64, 32),
                                 activation='relu',
                                 alpha=0.0001,
                                 max_iter=200,
                                 random_state=semilla_actual))
        ])

        pipeline.fit(X_train, y_train)
        y_pred_test = pipeline.predict(X_test)

        acc_test = accuracy_score(y_test, y_pred_test)
        recall_test = recall_score(y_test, y_pred_test)

        mlflow.set_tag("fase", "semillas")
        mlflow.log_params({"arquitectura": "(64, 32)", "activacion": "relu", "random_state": semilla_actual, "escalado": True})
        mlflow.log_metrics({"acc_test": acc_test, "recall_maligna": recall_test})

        resultados_semillas.append({'Semilla': semilla_actual, 'Accuracy': acc_test, 'Recall': recall_test})
        print(f"[{nombre_run}] | Acc Test: {acc_test:.4f} | Recall: {recall_test:.4f}")

# Cálculo de Mínimo, Máximo y Rango
df_semillas = pd.DataFrame(resultados_semillas)
print("\n--- Resumen de Variabilidad (5 Semillas) ---")
print(f"Accuracy -> Min: {df_semillas['Accuracy'].min():.4f} | Max: {df_semillas['Accuracy'].max():.4f} | Rango: {df_semillas['Accuracy'].max() - df_semillas['Accuracy'].min():.4f}")
print(f"Recall   -> Min: {df_semillas['Recall'].min():.4f} | Max: {df_semillas['Recall'].max():.4f} | Rango: {df_semillas['Recall'].max() - df_semillas['Recall'].min():.4f}")

# Fase 6: Replicación exacta del Run C original
with mlflow.start_run(run_name="Run_Replica_Exacta_C"):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', alpha=0.0001, max_iter=200, random_state=SEED))
    ])
    pipeline.fit(X_train, y_train)
    y_pred_test = pipeline.predict(X_test)

    mlflow.set_tag("fase", "replica")
    mlflow.log_metrics({"acc_test": accuracy_score(y_test, y_pred_test), "recall_maligna": recall_score(y_test, y_pred_test)})
    print(f"\n[Run_Replica_Exacta_C] Acc Test: {accuracy_score(y_test, y_pred_test):.4f} | Recall: {recall_score(y_test, y_pred_test):.4f}")

[Run_Semilla_33] | Acc Test: 0.9474 | Recall: 0.8810
🏃 View run Run_Semilla_33 at: http://127.0.0.1:5000/#/experiments/1/runs/26d35995073f44e18466b6a702dffece
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Semilla_34] | Acc Test: 0.9474 | Recall: 0.9286
🏃 View run Run_Semilla_34 at: http://127.0.0.1:5000/#/experiments/1/runs/e5586be9f10044fc9330d7ca2213fe9c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Semilla_35] | Acc Test: 0.9386 | Recall: 0.9048
🏃 View run Run_Semilla_35 at: http://127.0.0.1:5000/#/experiments/1/runs/bc0b206f09ac47cfa1efa544cb0d4e38
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Semilla_36] | Acc Test: 0.9386 | Recall: 0.9048
🏃 View run Run_Semilla_36 at: http://127.0.0.1:5000/#/experiments/1/runs/84bb01b0760a4c09950efdab8ccc45b2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Semilla_37] | Acc Test: 0.9298 | Recall: 0.8810
🏃 View run Run_Semilla_37 at: http://127.0.0.1:5000/#/experiments/1/run

### Interpretación de Resultados: Fase 6 - Semillas y Réplica Exacta

#### 1. Verificación de Réplica Exacta (Run_Replica_Exacta_C)
* **Replicabilidad determinista:** La corrida `Run_Replica_Exacta_C` (ejecutada con la semilla base `SEED = 33`) obtuvo exactamente un **Accuracy de 0.9474** y un **Recall de 0.8810**.
* **Coincidencia:** Al comparar estos valores con el `Run_C` original de la Fase 3 y con `Run_Semilla_33`, se observa que las métricas coinciden al 100% (diferencia de 0.0), lo que confirma la completa trazabilidad y reproductibilidad del modelo en el entorno de evaluación.

#### 2. Análisis de Variabilidad entre 5 Semillas (Run_C: 64, 32 + ReLU)
Al evaluar la misma arquitectura e hiperparámetros variando únicamente la semilla de inicialización del modelo (`SEED = 33` a `37`), se obtuvieron las siguientes métricas de dispersión:

* **Exactitud (Accuracy):**
  * **Mínimo:** 0.9298 (`Run_Semilla_37`)
  * **Máximo:** 0.9474 (`Run_Semilla_33` y `Run_Semilla_34`)
  * **Rango de variación:** 0.0175 (1.75%)
* **Sensibilidad (Recall - Clase Maligna):**
  * **Mínimo:** 0.8810 (`Run_Semilla_33` y `Run_Semilla_37`)
  * **Máximo:** 0.9286 (`Run_Semilla_34`)
  * **Rango de variación:** 0.0476 (4.76%)

#### 3. Conclusión sobre la Sensibilidad a la Inicialización
La variación observada demuestra el impacto del estado inicial aleatorio de los pesos en las redes neuronales. En particular, la sensibilidad (*Recall*) varió casi un **4.8%** entre semillas usando exactamente el mismo código y datos, lo que evidencia que un solo entrenamiento con una semilla específica no es suficiente para declarar una configuración como definitivamente superior a otra.

In [14]:
# Fase 7: Exportación de resultados a CSV
# Extraer todas las ejecuciones del experimento actual
runs_df = mlflow.search_runs()

# Seleccionar solo las columnas más relevantes para limpiar el archivo de salida
columnas_relevantes = [
    'run_id', 'tags.mlflow.runName', 'tags.fase',
    'params.arquitectura', 'params.activacion', 'params.escalado', 'params.random_state',
    'metrics.acc_test', 'metrics.recall_maligna', 'metrics.iteraciones'
]
# Filtrar columnas que existan en el DataFrame devuelto
runs_df = runs_df[[col for col in columnas_relevantes if col in runs_df.columns]]

# Exportar a CSV
archivo_csv = "runs.csv"
runs_df.to_csv(archivo_csv, index=False)
print(f"Archivo exportado con éxito: {archivo_csv}. Listo para descargar.")

Archivo exportado con éxito: runs.csv. Listo para descargar.


### Fase 8: Preguntas de Cierre

**1. ¿Qué cambió al escalar las variables? Respóndelo con evidencia de tus runs A y B y con el concepto de feature scaling.**
Al aplicar escalado (`Run_B`) el modelo mejoró significativamente en comparación con el `Run_A` (sin escalado). Esto ocurre porque el Perceptrón Multicapa se basa en el descenso del gradiente; sin *feature scaling*, variables con grandes magnitudes como el área del tumor dominan las sumas ponderadas de las neuronas, saturando las funciones de activación y ralentizando el aprendizaje. Al estandarizar, todas las características aportan de manera equilibrada a la frontera de decisión.

**2. ¿Más capas y neuronas mejoraron el resultado? ¿Cómo lo sabes, considerando la variación que observaste entre semillas?**
El `Run_C` (arquitectura 64,32) logró mejores métricas iniciales que el `Run_B` (30 neuronas). Sin embargo, al observar la variación entre semillas del paso 6, el rango de exactitud y *recall* demuestra que una red más compleja no garantiza siempre un mejor resultado general. La aparente superioridad de la arquitectura (64, 32) podría estar influenciada por la suerte de la inicialización de pesos en una semilla en particular.

**3. Con tus 5 semillas, ¿qué tan grande fue la variación de la exactitud y de la sensibilidad? ¿Es mayor o menor que las diferencias entre configuraciones? ¿Qué implica esto para afirmar que una configuración es “mejor”?**
La variación de exactitud y sensibilidad entre las 5 semillas fue notable (Rango de Accuracy: [Insertar Valor] | Rango de Recall: [Insertar Valor]). Esta variación interna por semilla es [mayor/menor] que el salto entre el Run B y C. Esto implica que no podemos afirmar categóricamente que una configuración es "mejor" basándonos en una sola ejecución; se requieren validaciones estadísticas (como promedios de múltiples semillas o *cross-validation*) para aislar el verdadero efecto de la arquitectura frente al azar.

**4. En el diagnóstico de cáncer, ¿qué error es más costoso: un falso negativo o un falso positivo? ¿Qué métrica priorizarías y por qué?**
En medicina, un **Falso Negativo** es muchísimo más costoso (clasificar un tumor maligno como benigno), ya que el paciente no recibiría tratamiento oportuno, arriesgando su vida. Por lo tanto, priorizaría la **Sensibilidad (Recall)** de la clase maligna, ya que esta métrica mide qué fracción de los tumores que *realmente* son malignos logró detectar el modelo, minimizando así los falsos negativos.

**5. ¿Qué información de MLflow no tendrías si solo hubieras usado `print()`? ¿Qué más registrarías para que otra persona pueda replicar tu experimento?**
Sin MLflow perdería la **observabilidad estructurada**: no tendría el historial consolidado en una interfaz visual comparativa, carecería del registro de la curva de aprendizaje por época (`train_loss` en cada *step*), y no tendría los gráficos (artefactos de matrices de confusión) vinculados permanentemente a los hiperparámetros que los generaron. Para una replicación perfecta, registraría además las versiones exactas de las librerías (`scikit-learn`, `pandas`, `python`) y la estructura del sistema operativo.

In [15]:
# Cierra el túnel de ngrok antes de finalizar
ngrok.kill()
print("Túnel cerrado. No olvides descargar el runs.csv y tus capturas de pantalla.")

Túnel cerrado. No olvides descargar el runs.csv y tus capturas de pantalla.
